# MOF CO₂ Uptake — R²-Optimized Supervised Pipelines (with Baselines, Cleaning, and Correlations)

This notebook follows a **consistent, reproducible pipeline** for the MOF CO₂ uptake task and **optimizes models by R²** (coefficient of determination). We still **report RMSE & MAE** for interpretability.

**What this notebook does**  
1. Load data (`train.csv`) and define **FEATURES** and **TARGET**  
2. **Clean**: remove duplicate rows and redundant/constant columns  
3. Quick **EDA** and **correlation analysis** (Pearson & Spearman)  
4. Create **stratified splits** (by binning the target) to regularize validation  
5. Build **dummy baselines** (mean/median)  
6. Set up **ColumnTransformer** preprocessing shared by all models  
7. Train/tune multiple **regression models** (Ridge, Lasso, ElasticNet, Kernel Ridge, SVR, RandomForest, + XGBoost/LightGBM/CatBoost if available) with **RandomizedSearchCV** optimizing **R²**  
8. Show a **leaderboard** ranked by R², also printing RMSE & MAE  
9. (Optional) Refit best model on full data for downstream use


In [3]:
import warnings, os, math, gc
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

from pathlib import Path

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, make_scorer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.dummy import DummyRegressor

# Linear / kernel / tree models
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.kernel_ridge import KernelRidge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

# Optional (if installed)
try:
    from xgboost import XGBRegressor
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False

try:
    from lightgbm import LGBMRegressor
    HAVE_LGBM = True
except Exception:
    HAVE_LGBM = False

try:
    from catboost import CatBoostRegressor
    HAVE_CAT = True
except Exception:
    HAVE_CAT = False

SEED = 42
N_SPLITS = 5
N_JOBS = -1
N_ITER = 40  # tuning iterations per model
PRIMARY_SCORER = "r2"  # optimize by R²
VERBOSE = 1

print({"xgb": HAVE_XGB, "lgbm": HAVE_LGBM, "catboost": HAVE_CAT})


SystemError: <built-in function isinstance> returned a result with an exception set

## 1. Load Data & Define `FEATURES` and `TARGET`


In [ ]:
from pathlib import Path
TRAIN_PATH = Path("/mnt/data/train.csv")
assert TRAIN_PATH.exists(), f"Cannot find {TRAIN_PATH}. Upload train.csv first."

df = pd.read_csv(TRAIN_PATH)
print("Shape:", df.shape)
display(df.head())

# Detect ID and TARGET
cols = list(df.columns)
id_candidates = [c for c in cols if c.lower() in ["id","sample_id","mof_id","material_id","index"] or c.lower().endswith("_id")]
target_candidates = [c for c in cols if c.lower() in [
    "target","y","label",
    "co2_uptake","co2 uptake","co2-uptake",
    "co2","uptake","uptake_co2",
    "co2_uptake_mmol_g","co2_uptake_mmol/g","co2_uptake_mmol_per_g","co2_uptake_mmol g"
]]

ID_COL = id_candidates[0] if id_candidates else None
TARGET = target_candidates[0] if target_candidates else None

if TARGET is None:
    numeric_cols_all = [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]
    TARGET = numeric_cols_all[-1] if numeric_cols_all else cols[-1]

FEATURES = [c for c in df.columns if c not in [ID_COL, TARGET]]

print("ID_COL:", ID_COL)
print("TARGET:", TARGET)
print("FEATURES (count):", len(FEATURES))


## 2. Cleaning — remove duplicates & redundant columns


In [ ]:
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]
print(f"Removed {before - after} duplicate rows. New shape: {df.shape}")

constant_cols = [c for c in FEATURES if df[c].nunique(dropna=False) <= 1]
if constant_cols:
    print("Dropping constant columns:", constant_cols)
    df = df.drop(columns=constant_cols)
    FEATURES = [c for c in FEATURES if c not in constant_cols]

to_drop = []
cols_list = list(FEATURES)
for i in range(len(cols_list)):
    c1 = cols_list[i]
    if c1 in to_drop: 
        continue
    for j in range(i+1, len(cols_list)):
        c2 = cols_list[j]
        if c2 in to_drop:
            continue
        if df[c1].equals(df[c2]):
            to_drop.append(c2)

if to_drop:
    print("Dropping duplicated columns:", to_drop)
    df = df.drop(columns=to_drop)
    FEATURES = [c for c in FEATURES if c not in to_drop]

print("Final FEATURES count after cleaning:", len(FEATURES))


## 3. Quick EDA & Target Distribution


In [ ]:
print("Info:")
df.info()

print("\\nDescribe (numeric):")
display(df.describe())

plt.figure(figsize=(6,4))
df[TARGET].hist(bins=40)
plt.title(f"Target distribution — {TARGET}")
plt.xlabel(TARGET); plt.ylabel("count")
plt.show()


## 4. Correlation Analysis (Pearson & Spearman)


In [ ]:
num_feats = [c for c in FEATURES if pd.api.types.is_numeric_dtype(df[c])]
corr_pearson = df[num_feats + [TARGET]].corr(method="pearson")
corr_spearman = df[num_feats + [TARGET]].corr(method="spearman")

print("Pearson correlation with target (top 20 by abs value):")
pearson_target = corr_pearson[TARGET].drop(labels=[TARGET]).abs().sort_values(ascending=False).head(20)
display(pearson_target)

print("Spearman correlation with target (top 20 by abs value):")
spearman_target = corr_spearman[TARGET].drop(labels=[TARGET]).abs().sort_values(ascending=False).head(20)
display(spearman_target)

def plot_corr_heatmap(corr, title):
    plt.figure(figsize=(8,6))
    plt.imshow(corr, aspect='auto', interpolation='nearest')
    plt.colorbar()
    plt.title(title)
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=6)
    plt.yticks(range(len(corr.index)), corr.index, fontsize=6)
    plt.tight_layout()
    plt.show()

plot_corr_heatmap(corr_pearson, "Pearson Correlation Heatmap (numeric + target)")
plot_corr_heatmap(corr_spearman, "Spearman Correlation Heatmap (numeric + target)")


## 5. Stratified Train/Validation Split (by binned target)


In [ ]:
from sklearn.model_selection import train_test_split

def make_strata(y, q=10):
    import pandas as pd
    try:
        return pd.qcut(y, q=q, duplicates="drop")
    except Exception:
        return pd.qcut(pd.Series(y).rank(method="first"), q=q, duplicates="drop")

strata = make_strata(df[TARGET].values, q=10)
X = df[FEATURES].copy()
y = df[TARGET].values

X_train, X_valid, y_train, y_valid, strata_train, strata_valid = train_test_split(
    X, y, strata, test_size=0.2, random_state=42, stratify=strata
)
print("Train/Valid shapes:", X_train.shape, X_valid.shape)


## 6. Preprocessing with ColumnTransformer (shared across all models)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

numeric_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(df[c])]
categorical_cols = [c for c in X.columns if not pd.api.types.is_numeric_dtype(df[c])]

numeric_proc = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

categorical_proc = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_proc, numeric_cols),
        ("cat", categorical_proc, categorical_cols),
    ],
    remainder="drop",
    n_jobs=-1
)
print("Numeric/Categorical:", len(numeric_cols), len(categorical_cols))


## 7. Baseline (Dummy) Models


In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

baselines = {
    "DummyMean": Pipeline([("prep", preprocess), ("model", DummyRegressor(strategy="mean"))]),
    "DummyMedian": Pipeline([("prep", preprocess), ("model", DummyRegressor(strategy="median"))]),
}

for name, pipe in baselines.items():
    pipe.fit(X_train, y_train)
    p = pipe.predict(X_valid)
    r2 = r2_score(y_valid, p)
    rmse = mean_squared_error(y_valid, p, squared=False)
    mae = mean_absolute_error(y_valid, p)
    print(f"{name:>12s} | R2={r2:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}")


## 8. Model Factories with Hyperparameter Spaces (focus on regularization)


In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.kernel_ridge import KernelRidge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

import numpy as np

def make_ridge():
    pipe = Pipeline([("prep", preprocess), ("model", Ridge(random_state=42))])
    params = {"model__alpha": np.logspace(-4, 4, 60)}
    return "Ridge", pipe, params

def make_lasso():
    pipe = Pipeline([("prep", preprocess), ("model", Lasso(random_state=42, max_iter=10000))])
    params = {"model__alpha": np.logspace(-5, 2, 60)}
    return "Lasso", pipe, params

def make_enet():
    pipe = Pipeline([("prep", preprocess), ("model", ElasticNet(random_state=42, max_iter=10000))])
    params = {"model__alpha": np.logspace(-5, 2, 60), "model__l1_ratio": np.linspace(0.0, 1.0, 21)}
    return "ElasticNet", pipe, params

def make_krr():
    pipe = Pipeline([("prep", preprocess), ("model", KernelRidge())])
    params = {
        "model__alpha": np.logspace(-4, 3, 40),
        "model__kernel": ["rbf","laplacian","poly","sigmoid"],
        "model__gamma": np.logspace(-5, 1, 30),
        "model__degree": [2,3,4],
        "model__coef0": np.linspace(0.0, 2.0, 11),
    }
    return "KernelRidge", pipe, params

def make_svr():
    pipe = Pipeline([("prep", preprocess), ("model", SVR())])
    params = {
        "model__kernel": ["rbf","poly","sigmoid"],
        "model__C": np.logspace(-2, 3, 30),
        "model__gamma": np.logspace(-5, 1, 20),
        "model__epsilon": np.logspace(-4, -1, 8),
        "model__degree": [2,3,4],
        "model__coef0": np.linspace(0, 2, 9),
    }
    return "SVR", pipe, params

def make_rf():
    pipe = Pipeline([("prep", preprocess), ("model", RandomForestRegressor(random_state=42))])
    params = {
        "model__n_estimators": [400, 800, 1200],
        "model__max_depth": [None, 8, 12, 16, 24],
        "model__min_samples_split": [2, 5, 10, 20],
        "model__min_samples_leaf": [1, 2, 4, 8],
        "model__max_features": ["sqrt", "log2", None],
        "model__bootstrap": [True, False],
    }
    return "RandomForest", pipe, params

def make_extratrees():
    pipe = Pipeline([("prep", preprocess), ("model", ExtraTreesRegressor(random_state=42))])
    params = {
        "model__n_estimators": [600, 1000, 1400],
        "model__max_depth": [None, 8, 12, 16],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", "log2", None],
    }
    return "ExtraTrees", pipe, params

# Optional
try:
    from xgboost import XGBRegressor
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False

def make_xgb():
    if not HAVE_XGB: return None
    pipe = Pipeline([("prep", preprocess), ("model", XGBRegressor(random_state=42, tree_method="hist"))])
    params = {
        "model__n_estimators": [800, 1200, 1600],
        "model__learning_rate": [0.02, 0.03, 0.05],
        "model__max_depth": [4,6,8,10],
        "model__min_child_weight": [1,3,5],
        "model__subsample": [0.7,0.8,1.0],
        "model__colsample_bytree": [0.7,0.8,1.0],
        "model__reg_alpha": [0, 0.1, 0.5, 1.0],
        "model__reg_lambda": [0.5, 1.0, 2.0],
    }
    return "XGBoost", pipe, params

factories = [make_ridge, make_lasso, make_enet, make_krr, make_svr, make_rf, make_extratrees, make_xgb]
models = [f() for f in factories]
models = [m for m in models if m is not None]
print("Models:", [m[0] for m in models])


## 9. Tuning Helper (optimize by R²) and Evaluation (also show RMSE & MAE)


In [ ]:
from sklearn.model_selection import KFold, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

cv = KFold(n_splits=5, shuffle=True, random_state=42)

def tune_and_eval(label, pipe, param_dist, X, y, cv, n_iter=40):
    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring="r2",
        cv=cv,
        n_jobs=-1,
        random_state=42,
        verbose=1,
        refit=True
    )
    search.fit(X, y)
    best_pipe = search.best_estimator_
    y_pred_cv = cross_val_predict(best_pipe, X, y, cv=cv, n_jobs=-1, verbose=0)
    r2_cv  = r2_score(y, y_pred_cv)
    rmse_cv = mean_squared_error(y, y_pred_cv, squared=False)
    mae_cv = mean_absolute_error(y, y_pred_cv)

    best_pipe.fit(X_train, y_train)
    y_val_pred = best_pipe.predict(X_valid)
    r2_val  = r2_score(y_valid, y_val_pred)
    rmse_val = mean_squared_error(y_valid, y_val_pred, squared=False)
    mae_val = mean_absolute_error(y_valid, y_val_pred)

    print(f"\\n[{label}] Best CV R2={search.best_score_:.5f}")
    print("  Best params:", search.best_params_)
    print(f"  CV (R2/RMSE/MAE): {r2_cv:.5f} / {rmse_cv:.5f} / {mae_cv:.5f}")
    print(f"  Hold-out (R2/RMSE/MAE): {r2_val:.5f} / {rmse_val:.5f} / {mae_val:.5f}")

    return {
        "model": label,
        "best_cv_r2": search.best_score_,
        "cv_rmse": rmse_cv,
        "cv_mae": mae_cv,
        "holdout_r2": r2_val,
        "holdout_rmse": rmse_val,
        "holdout_mae": mae_val,
        "best_params": search.best_params_,
        "fitted": best_pipe,
        "search_obj": search
    }


## 10. Run All Models & Build R² Leaderboard


In [ ]:
results = []
for label, pipe, params in models:
    res = tune_and_eval(label, pipe, params, X, y, cv=cv, n_iter=40)
    results.append(res)

leaderboard = pd.DataFrame([
    {"model": r["model"], "R2_CV": r["best_cv_r2"], "RMSE_CV": r["cv_rmse"], "MAE_CV": r["cv_mae"],
     "R2_Holdout": r["holdout_r2"], "RMSE_Holdout": r["holdout_rmse"], "MAE_Holdout": r["holdout_mae"],
     "best_params": r["best_params"]}
    for r in results
]).sort_values("R2_CV", ascending=False).reset_index(drop=True)

display(leaderboard)


## 11. Refit Best Model on Full Data (optional)


In [ ]:
best = max(results, key=lambda d: d["best_cv_r2"])
print("Best by CV R2:", best["model"], best["best_cv_r2"])
final_model = best["fitted"]
final_model.fit(X, y)
# import joblib; joblib.dump(final_model, "best_r2_pipeline.pkl")
